In [ ]:
import dill
import numpy as np
from utili.misc.misc import adjust_accelerator_settings, create_result_folder, compute_output_padding
from utili.eval_funs import eval_metrics, inference_map, plot_confusion
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
import Dataset.FLAIR as Dataset

### Purpose:
Load models trained using the **train** notebook.

### Cell Dependency:
All cells above.

In [ ]:
#Interface


adapted_models_path = None
assert adapted_models_path is not None, "adapted_models_path cannot be None"
with open(adapted_models_path, 'rb') as file:
    t_model_my = dill.load(file)

### Purpose:
Load Our pre-trained models from HuggingFace.

### Cell Dependency:
The first two cells

In [ ]:
from huggingface_hub import snapshot_download
import os

repo_id = "anan1966752/GeoAI-FoundGen_models_pt"

hf_dir = os.path.join(os.getcwd(), "HF")

if not os.path.exists(hf_dir):
    os.makedirs(hf_dir)

subfolder = Dataset.Dataset_name

path = snapshot_download(repo_id=repo_id, repo_type="model", local_dir_use_symlinks=False, local_dir=hf_dir)

sub_path = os.path.join(path, subfolder)

t_model_my = Dataset.load_from_state_dict(Dataset, path=sub_path)

### Purpose:
Per Run Evaluation Metrics

### Cell Dependency:
All cells above

In [ ]:
t_cm_my = []
t_acc_my = []
t_miou_my = []
t_mf1_my = []
t_f1_my = []
gpu_num = 0


da_datamodule = Dataset.DA_Datamodule(num_workers=16)


for idx, t_cnn in enumerate(t_model_my):
    temp_acc, temp_cm, temp_miou, temp_mf1, union_unique_classes_contigous, temp_f1 = eval_metrics(
        t_cnn.eval().cuda(), "Target My", da_datamodule.Target_test_dataloader, DA_Datamodule=da_datamodule,
    )
    t_acc_my.append(temp_acc)
    t_cm_my.append(temp_cm[..., None])
    t_miou_my.append(temp_miou)
    t_mf1_my.append(temp_mf1)
    t_f1_my.append(temp_f1)

### Purpose:
Per Experiment Evaluation Metrics

### Cell Dependency:
All cells above

In [ ]:
print("T my Acc mean " + str(np.mean(t_acc_my)))
print("T my Acc std " + str(np.sqrt(np.var(t_acc_my))))
print('T my mIoU mean ' + str(np.mean(t_miou_my)))
print('T my mIoU std' + str(np.var(t_miou_my)))
print('T mF1 mean ' + str(np.mean(t_mf1_my)))
print('T mF1 std' + str(np.var(t_mf1_my)))
a, all_f_t = plot_confusion(
    t_cm_my, "T my ", DA_Datamodule=da_datamodule, list_f1=t_f1_my,
    union_classes=union_unique_classes_contigous,
)

### Purpose:
Inference maps generation

### Cell Dependency:
The first 4 cells above

In [ ]:
inference_map(
    T_cnn=t_model_my[0].eval(),
    Target_test_dataloader=da_datamodule.Target_test_dataloader,
    DA_Datamodule=da_datamodule,
    name="Run_0",
    GPU_NUM=gpu_num,
)